In [2]:
import os
import glob
import cv2
import wandb
import torch
from ultralytics import YOLO
import matplotlib.pyplot as plt


DATA_DIR = "/home/nagaraj/Garbage_classification_files/Garbage classification"
OUTPUT_DIR = "partC_results"
PROJECT = "Atri"
ENTITY = "cs24s023-iitm-ac-in"
os.makedirs(OUTPUT_DIR, exist_ok=True)

wandb.init(project=PROJECT, entity=ENTITY, name="PartC_YOLO_Demo", reinit=True)


classes = sorted(os.listdir(os.path.join(DATA_DIR, "train")))
class_to_idx = {cls: i for i, cls in enumerate(classes)}

yolo_dir = os.path.join(OUTPUT_DIR, "yolo_dataset")
for split in ["train", "test"]:
    img_dir = os.path.join(yolo_dir, "images", split)
    lbl_dir = os.path.join(yolo_dir, "labels", split)
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    for cls in classes:
        img_paths = glob.glob(os.path.join(DATA_DIR, split, cls, "*.jpg"))
        for img_path in img_paths:
            fname = os.path.basename(img_path)
            dst_img = os.path.join(img_dir, fname)
            dst_lbl = os.path.join(lbl_dir, fname.replace(".jpg", ".txt"))
            os.system(f"cp '{img_path}' '{dst_img}'")
            with open(dst_lbl, "w") as f:
                f.write(f"{class_to_idx[cls]} 0.5 0.5 1.0 1.0\n")

with open(os.path.join(OUTPUT_DIR, "garbage.yaml"), "w") as f:
    f.write(f"path: {yolo_dir}\n")
    f.write("train: images/train\n")
    f.write("val: images/test\n")
    f.write(f"names: {classes}\n")


model = YOLO("yolov8n.pt")
model.train(data=os.path.join(OUTPUT_DIR, "garbage.yaml"),
            epochs=30, imgsz=224, project=OUTPUT_DIR, name="yolo_garbage")


metrics = model.val()
wandb.log({"mAP50": metrics.box.map50, "mAP": metrics.box.map})


def run_inference_and_save(split, nmax=50):
    img_dir = os.path.join(yolo_dir, "images", split)
    out_dir = os.path.join(OUTPUT_DIR, f"{split}_detected")
    os.makedirs(out_dir, exist_ok=True)

    img_paths = glob.glob(os.path.join(img_dir, "*.jpg"))[:nmax]
    results = model.predict(img_paths, save=True, project=out_dir, name="preds", imgsz=224)

    
    annotated = list(glob.glob(os.path.join(out_dir, "preds", "*.jpg")))
    return annotated

train_imgs = run_inference_and_save("train", nmax=200)
test_imgs = run_inference_and_save("test", nmax=200)


def make_video(img_list, out_path, fps=5):
    if not img_list:
        print(f" No images for {out_path}")
        return
    frame = cv2.imread(img_list[0])
    h, w, _ = frame.shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video = cv2.VideoWriter(out_path, fourcc, fps, (w, h))
    for img in img_list:
        frame = cv2.imread(img)
        video.write(frame)
    video.release()
    print(f" Saved video: {out_path}")

train_video = os.path.join(OUTPUT_DIR, "train_detected.mp4")
test_video = os.path.join(OUTPUT_DIR, "test_detected.mp4")
make_video(train_imgs, train_video)
make_video(test_imgs, test_video)


wandb.log({
    "train_detected_video": wandb.Video(train_video, fps=5, format="mp4"),
    "test_detected_video": wandb.Video(test_video, fps=5, format="mp4"),
    "sample_detections": [wandb.Image(img) for img in test_imgs[:10]]
})

wandb.finish()

print(" Part C complete! Check W&B dashboard + local videos in:", OUTPUT_DIR)


New https://pypi.org/project/ultralytics/8.3.203 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.201 🚀 Python-3.11.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3080, 9875MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=partC_results/garbage.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo_garbage, nbs=64, nm

wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.
wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.
wandb: ERROR The nbformat package was not found. It is required to save notebook history.


 Saved video: partC_results/test_detected.mp4


mAP,▁
mAP50,▁
mAP,0.94586
mAP50,0.94586


 Part C complete! Check W&B dashboard + local videos in: partC_results
